# Solution 15: Full TTA on Solution 14

No retraining. I applied test-time augmentation by running all N! choice permutations for each question through the Solution 14 adapter, mapping logits back to original positions, and averaging.

| num_choices | permutations per question |
|-------------|--------------------------|
| 2           | 2                        |
| 3           | 6                        |
| 4           | 24                       |
| 5           | 120                      |

Big improvement from inference alone: 0.92354, up from 0.91549.

**Score: 0.92354**

## 0. Install Dependencies

In [1]:
!pip install -q "transformers==4.47.0"

!pip uninstall -y torchao 2>/dev/null

!pip install -q accelerate peft bitsandbytes datasets pillow tqdm pandas

import transformers, peft
print(f"transformers: {transformers.__version__}")
print(f"peft: {peft.__version__}")
print("All good. If first run, do Runtime -> Restart session, then run all cells.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 139.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 132.5 MB/s eta 0:00:00
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 46.9 MB/s eta 0:00:00
transformers: 4.47.0
peft: 0.19.1
All good. If first run, do Runtime -> Restart session, then run all cells.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Imports & Configuration

In [6]:
import transformers
print(f"transformers: {transformers.__version__}")
assert transformers.__version__ >= "4.45.0", f"Too old: {transformers.__version__}"

import os, json, random, math
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from itertools import permutations

import torch
from transformers import AutoProcessor, AutoModelForVision2Seq
from peft import PeftModel

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
DATA_DIR = "/content/drive/MyDrive/pixels-to-predictions"
IMG_BASE = os.path.join(DATA_DIR, "images")
MAX_SEQ_LEN = 1024
IMAGE_LONGEST_EDGE = 512

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

transformers: 4.47.0
Device: cuda
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


## Load Data (Val & Test Only)

In [7]:
val_df = pd.read_csv(os.path.join(DATA_DIR, "val.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
print(f"Val: {len(val_df)} | Test: {len(test_df)}")

print(f"\nVal num_choices distribution:")
print(val_df['num_choices'].value_counts().sort_index())
print(f"\nTest num_choices distribution:")
print(test_df['num_choices'].value_counts().sort_index())

val_passes = sum(math.factorial(nc) for nc in val_df['num_choices'])
test_passes = sum(math.factorial(nc) for nc in test_df['num_choices'])
print(f"\nTotal TTA forward passes — Val: {val_passes:,} | Test: {test_passes:,}")

Val: 1048 | Test: 1008

Val num_choices distribution:
num_choices
2    244
3    508
4    252
5     44
Name: count, dtype: int64

Test num_choices distribution:
num_choices
2    272
3    438
4    260
5     38
Name: count, dtype: int64

Total TTA forward passes — Val: 14,864 | Test: 13,972


## 3. Load Model & Processor

In [8]:
adapter_candidates = [
    "/content/smolvlm-lora-v16",
    "/content/drive/MyDrive/pixels-to-predictions/checkpoints_v16/epoch_5",
    "/content/drive/MyDrive/pixels-to-predictions/checkpoints_v16/epoch_6",
    "/content/drive/MyDrive/pixels-to-predictions/checkpoints_v16/lora_adapter",
]

adapter_path = None
for path in adapter_candidates:
    if os.path.exists(os.path.join(path, "adapter_config.json")):
        adapter_path = path
        break

if adapter_path is None:
    raise FileNotFoundError(f"No v16 adapter found! Checked: {adapter_candidates}")

processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.image_processor.size = {"longest_edge": IMAGE_LONGEST_EDGE}

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

print(f"Loaded v16 adapter from: {adapter_path}")
print(f"Image size: {processor.image_processor.size}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Some kwargs in processor config are unused and will not have any effect: image_seq_len. 


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Loaded v16 adapter from: /content/drive/MyDrive/pixels-to-predictions/checkpoints_v16/epoch_5
Image size: {'longest_edge': 512}


In [9]:
def build_prompt(row, include_answer=False):
    """Identical to v16."""
    choices = json.loads(row['choices'])
    choices_text = "\n".join(f"({i}) {c}" for i, c in enumerate(choices))

    parts = []
    if pd.notna(row.get('hint', None)) and str(row['hint']).strip():
        parts.append(f"Context: {row['hint'].strip()}")
    if pd.notna(row.get('lecture', None)) and str(row['lecture']).strip():
        parts.append(f"Background: {row['lecture'].strip()}")

    parts.append(f"Question: {row['question'].strip()}")
    parts.append(f"Choices:\n{choices_text}")
    parts.append("Answer with ONLY the number of the correct choice.")

    prompt_text = "\n\n".join(parts)

    if include_answer:
        answer_text = str(int(row['answer']))
        return prompt_text, answer_text
    return prompt_text

sample = val_df.iloc[0]
prompt, answer = build_prompt(sample, include_answer=True)
print("SAMPLE PROMPT (first 400 chars):")
print(prompt[:400])
print(f"\n>>> TARGET ANSWER: {answer}")

SAMPLE PROMPT (first 400 chars):
Context: Animals often behave in certain ways that can increase their reproductive success. Read the passage about a specific animal behavior. Then, follow the instructions below.

The snail leech is a type of worm that often lives in freshwater streams. After  reproduces, it attaches its eggs to a rock at the bottom of the stream. The leech then flattens its body over its eggs to protect them. Th

>>> TARGET ANSWER: 0


In [10]:
DIGIT_TOKEN_IDS = [processor.tokenizer.encode(str(d), add_special_tokens=False)[0] for d in range(10)]
print(f"Digit token IDs: { {d: DIGIT_TOKEN_IDS[d] for d in range(5)} }")

@torch.no_grad()
def predict_single_pass(model, processor, df, img_base, batch_size=4):
    """Standard single-pass digit logit prediction (same as v16)."""
    model.eval()
    all_ids, all_preds = [], []

    for start in tqdm(range(0, len(df), batch_size), desc="Single-pass"):
        batch_df = df.iloc[start:start + batch_size]
        images, texts, batch_ids, nc_list = [], [], [], []

        for _, row in batch_df.iterrows():
            images.append(Image.open(os.path.join(img_base, row['image_path'])).convert("RGB"))
            prompt = build_prompt(row, include_answer=False)
            messages = [{"role": "user", "content": [
                {"type": "image"}, {"type": "text", "text": prompt}]}]
            texts.append(processor.apply_chat_template(messages, add_generation_prompt=True))
            batch_ids.append(row['id'])
            nc_list.append(row['num_choices'])

        inputs = processor(text=texts, images=images, return_tensors="pt",
                           padding=True, truncation=True, max_length=MAX_SEQ_LEN).to(device)

        with torch.amp.autocast("cuda", dtype=torch.float16):
            outputs = model(**inputs)
        logits = outputs.logits

        for i in range(len(batch_df)):
            last_pos = inputs["attention_mask"][i].sum().item() - 1
            next_logits = logits[i, last_pos, :]
            nc = nc_list[i]
            scores = torch.stack([next_logits[DIGIT_TOKEN_IDS[d]] for d in range(nc)])
            all_preds.append(scores.argmax().item())
            all_ids.append(batch_ids[i])

    return all_ids, all_preds

@torch.no_grad()
def predict_tta(model, processor, df, img_base, batch_size=4):
    """TTA: run all N! choice permutations per question, average digit logits
    mapped back to original positions, pick the highest-scoring choice.

    For each permutation (e.g., perm = (2,0,3,1) for 4 choices):
      - Choices are reordered: [C, A, D, B]
      - Model sees them as (0)C, (1)A, (2)D, (3)B
      - Logit for digit d maps to original choice perm[d]
      - So we accumulate: original_scores[perm[d]] += logit[d]
    """
    model.eval()
    all_ids, all_preds = [], []

    for idx in tqdm(range(len(df)), desc="TTA"):
        row = df.iloc[idx]
        nc = row['num_choices']
        choices = json.loads(row['choices'])

        all_perms = list(permutations(range(nc)))

        original_scores = torch.zeros(nc, device=device)
        image = Image.open(os.path.join(img_base, row['image_path'])).convert("RGB")

        for perm_start in range(0, len(all_perms), batch_size):
            perm_batch = all_perms[perm_start:perm_start + batch_size]
            images, texts, perms = [], [], []

            for perm in perm_batch:
                perm_row = row.copy()
                perm_row['choices'] = json.dumps([choices[perm[i]] for i in range(nc)])
                prompt = build_prompt(perm_row, include_answer=False)
                messages = [{"role": "user", "content": [
                    {"type": "image"}, {"type": "text", "text": prompt}]}]
                texts.append(processor.apply_chat_template(messages, add_generation_prompt=True))
                images.append(image.copy())
                perms.append(perm)

            inputs = processor(text=texts, images=images, return_tensors="pt",
                             padding=True, truncation=True, max_length=MAX_SEQ_LEN).to(device)

            with torch.amp.autocast("cuda", dtype=torch.float16):
                outputs = model(**inputs)
            logits = outputs.logits

            for i, perm in enumerate(perms):
                last_pos = inputs["attention_mask"][i].sum().item() - 1
                next_logits = logits[i, last_pos, :]
                digit_scores = torch.stack([next_logits[DIGIT_TOKEN_IDS[d]] for d in range(nc)])
                for d in range(nc):
                    original_scores[perm[d]] += digit_scores[d]

        pred = original_scores.argmax().item()
        all_preds.append(pred)
        all_ids.append(row['id'])

    return all_ids, all_preds

print("Prediction functions defined.")

Digit token IDs: {0: 32, 1: 33, 2: 34, 3: 35, 4: 36}
Prediction functions defined.


## Validation: Single Pass vs TTA

In [11]:
print("=== Single-Pass Prediction (v16 baseline) ===")
val_ids_sp, val_preds_sp = predict_single_pass(model, processor, val_df, IMG_BASE, batch_size=4)
val_acc_sp = np.mean([p == a for p, a in zip(val_preds_sp, val_df['answer'].tolist())])
print(f"Single-pass Val Accuracy: {val_acc_sp:.4f}")

print("\n=== TTA Prediction (all permutations) ===")
val_ids_tta, val_preds_tta = predict_tta(model, processor, val_df, IMG_BASE, batch_size=4)
val_acc_tta = np.mean([p == a for p, a in zip(val_preds_tta, val_df['answer'].tolist())])
print(f"TTA Val Accuracy: {val_acc_tta:.4f}")

print(f"\n{'='*50}")
print(f"Single-pass: {val_acc_sp:.4f}")
print(f"TTA:         {val_acc_tta:.4f}")
print(f"Delta:       {val_acc_tta - val_acc_sp:+.4f}")
print(f"v16 Kaggle:  0.91549")

changed = sum(1 for sp, tta in zip(val_preds_sp, val_preds_tta) if sp != tta)
print(f"\nPredictions changed by TTA: {changed}/{len(val_df)} ({100*changed/len(val_df):.1f}%)")

fixes, breaks = 0, 0
for sp, tta, true_ans in zip(val_preds_sp, val_preds_tta, val_df['answer'].tolist()):
    if sp != tta:
        if tta == true_ans:
            fixes += 1
        elif sp == true_ans:
            breaks += 1
print(f"  Fixes (wrong->right): {fixes}")
print(f"  Breaks (right->wrong): {breaks}")
print(f"  Net improvement: {fixes - breaks}")

print(f"\nAccuracy by num_choices:")
for nc in [2, 3, 4, 5]:
    mask = val_df['num_choices'] == nc
    true = val_df.loc[mask, 'answer'].tolist()
    sp = [val_preds_sp[i] for i in range(len(val_df)) if mask.iloc[i]]
    tta = [val_preds_tta[i] for i in range(len(val_df)) if mask.iloc[i]]
    acc_sp = np.mean([p == a for p, a in zip(sp, true)])
    acc_tta = np.mean([p == a for p, a in zip(tta, true)])
    print(f"  {nc}-choice: SP={acc_sp:.4f} TTA={acc_tta:.4f} delta={acc_tta-acc_sp:+.4f}")

=== Single-Pass Prediction (v16 baseline) ===


Single-pass:   0%|          | 0/262 [00:00<?, ?it/s]

Single-pass Val Accuracy: 0.8979

=== TTA Prediction (all permutations) ===


TTA:   0%|          | 0/1048 [00:00<?, ?it/s]

TTA Val Accuracy: 0.9065

Single-pass: 0.8979
TTA:         0.9065
Delta:       +0.0086
v16 Kaggle:  0.91549

Predictions changed by TTA: 39/1048 (3.7%)
  Fixes (wrong->right): 21
  Breaks (right->wrong): 12
  Net improvement: 9

Accuracy by num_choices:
  2-choice: SP=0.8934 TTA=0.8770 delta=-0.0164
  3-choice: SP=0.9350 TTA=0.9390 delta=+0.0039
  4-choice: SP=0.8889 TTA=0.9286 delta=+0.0397
  5-choice: SP=0.5455 TTA=0.5682 delta=+0.0227


## Test Predictions (TTA) + Submission

In [12]:
print("Generating test predictions with TTA...")
test_ids_tta, test_preds_tta = predict_tta(model, processor, test_df, IMG_BASE, batch_size=4)
print(f"\nPrediction distribution: {pd.Series(test_preds_tta).value_counts().sort_index().to_dict()}")

submission = pd.DataFrame({"id": test_ids_tta, "answer": test_preds_tta})
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
assert set(submission['id']) == set(sample_sub['id']), "ID mismatch!"
submission = submission.set_index('id').loc[sample_sub['id']].reset_index()
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv (TTA)")
print(submission.head(10))

Generating test predictions with TTA...


TTA:   0%|          | 0/1008 [00:00<?, ?it/s]


Prediction distribution: {0: 364, 1: 361, 2: 208, 3: 69, 4: 6}
Saved submission.csv (TTA)
           id  answer
0  test_01750       2
1  test_00128       0
2  test_02891       3
3  test_02425       1
4  test_00930       2
5  test_03725       2
6  test_00009       0
7  test_02880       0
8  test_01208       0
9  test_00619       1


## Download

In [13]:
from google.colab import files
files.download('submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>